<a href="https://colab.research.google.com/github/Sagaustus/adh-group-projects/blob/main/group-04-unesco-heritage/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Five of One Hundred and Fifteen

### The Cultural/Natural binary and what it makes unsayable

**Group 4 · working chapter draft**

---

This notebook runs the analysis end to end. Cells marked **YOUR DECISION** are where
your judgement enters.

**The thesis you are testing.** The World Heritage List sorts every site into
Cultural, Natural, or — rarely — Mixed. For landscapes where sacred grove, farmland,
settlement and forest are one continuous practice, that binary is not a simplification
but a category error. Only five of 115 African sites are permitted to be both.

**A warning before you start.** Your dataset has **115 rows**. That is small. Several
methods you have been taught will produce confident-looking output on it that means
nothing, and this notebook stops you at each one.

In [ ]:
# Setup — run this first. Nothing to upload.
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

URL = "https://raw.githubusercontent.com/Sagaustus/adh-dh-datasets/main/datasets/11_unesco_heritage_africa/data.csv"
df = pd.read_csv(URL)
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(df.dtypes.to_string())

## Step 1 · Frame

| | Question | Method |
|---|---|---|
| **Descriptive** | Which inscription criteria carry African World Heritage sites, and how are sites distributed across states? | Frequency, concentration |
| **Analytical** | Did the 1994 Global Strategy change what gets inscribed from Africa? | Cross-tabulation before and after, with a test chosen for small counts |

The analytical question is good because UNESCO **made a specific promise** in 1994 —
the Global Strategy for a Representative, Balanced and Credible World Heritage List,
adopted precisely because the List over-represented European monumental heritage. A
promise with a date is a testable claim.

## Step 2 · Absence audit — the hard version

Every other group starts by finding empty cells. Run this and see what you get.

In [ ]:
print("MISSING VALUES IN THIS DATASET\n")
miss = df.isna().sum()
print(miss.to_string())
print(f"\ntotal missing cells: {int(df.isna().sum().sum())} of {df.size:,}")
print()
print("Zero. Not one blank cell anywhere.")
print()
print("Do NOT record that as a data-quality virtue and move on. It is the most")
print("interesting fact in the file, and it needs explaining rather than praising.")

### Why a dataset with no gaps should make you suspicious

A World Heritage nomination is a **dossier**. It is refused until every required field
is complete. So completeness here is not an observation about the world — it is an
entry condition, manufactured by the standard.

That inverts the absence audit. You cannot ask what the archive failed to record. You
have to ask **what the schema cannot express**, which is a harder and more interesting
question.

In [ ]:
print("WHAT THIS SCHEMA CANNOT EXPRESS\n")
for item, why in [
    ("a contested inscription",
     "no field records objection by a state, community or expert body"),
    ("a site that was nominated and refused",
     "the list contains only successes; failure leaves no trace here"),
    ("a community's refusal to be inscribed",
     "inscription is assumed to be desired"),
    ("degrees of 'mixed'",
     "category is a three-valued choice, not a proportion"),
    ("change in status over time",
     "one year_inscribed per site; In Danger listings are absent"),
    ("who wrote the nomination",
     "dossiers are authored, often by consultants; no field records it"),
]:
    print(f"   {item}")
    print(f"      -> {why}\n")

print(f"category is a three-valued field: {df['category'].value_counts().to_dict()}")
print()
print("YOUR DECISION. Which of these absences most limits your questions?")
print("The chapter argues one of them matters more than the rest — choose it now.")

## Step 3 · Describe

The List, as it stands for Africa.

In [ ]:
print(f"{len(df)} sites across {df['country'].nunique()} countries, "
      f"inscribed {df['year_inscribed'].min()}-{df['year_inscribed'].max()}\n")

print("by category")
for cat, n in df["category"].value_counts().items():
    print(f"   {cat:<12}{n:>5}  ({n/len(df):>5.1%})")
print()

per_country = df["country"].value_counts()
print("by country")
print(f"   mean sites per country: {len(df)/df['country'].nunique():.2f}")
print(f"   countries with exactly one site: {int((per_country==1).sum())} of {len(per_country)}")
print(f"   top five: {per_country.head(5).to_dict()}")
print()
print("Twenty-one states hold a single site each while Ethiopia holds twelve.")
print("Any per-country rate you compute will be unstable — see Step 5.")
print()
print(f"transnational sites: {int(df['transnational'].sum())}")
print(f"states per site: {df['n_states'].value_counts().to_dict()}")

In [ ]:
import re
from collections import Counter

def parse_criteria(c):
    """UNESCO criteria are roman numerals in parentheses, e.g. '(ix)(x)'."""
    return re.findall(r"\b(x|ix|viii|vii|vi|v|iv|iii|ii|i)\b", str(c).lower())

df["crit"] = df["criteria"].map(parse_criteria)
df["n_criteria"] = df["crit"].map(len)

counts = Counter(c for cs in df["crit"] for c in cs)
ORDER = ["i","ii","iii","iv","v","vi","vii","viii","ix","x"]
MEANING = {
 "i":"human creative genius", "ii":"interchange of human values",
 "iii":"testimony to a cultural tradition", "iv":"type of building or landscape",
 "v":"traditional human settlement", "vi":"association with events or beliefs",
 "vii":"superlative natural phenomena", "viii":"earth's history",
 "ix":"ecological processes", "x":"biodiversity and threatened species",
}
print(f"{'criterion':<12}{'sites':>7}   what it claims")
print("-" * 66)
for c in ORDER:
    print(f"   ({c}){'':<{7-len(c)}}{counts.get(c,0):>7}   {MEANING[c]}")
print()
print(f"criteria per site: mean {df['n_criteria'].mean():.2f}, "
      f"range {df['n_criteria'].min()}-{df['n_criteria'].max()}")
print(f"distinct criteria combinations: {df['criteria'].nunique()}")

### A trap: do not compare criteria against category

It is tempting to test whether Cultural sites use different criteria from Natural
ones. **They do, by definition.** Criteria (i)–(vi) are the cultural criteria and
(vii)–(x) the natural ones; the category is derived from which were used.

Testing that relationship would produce an enormous, meaningless effect — the same
circularity Group 2 meets in a different form. Look for a comparison that is not
built into the definitions, which is what Step 4 does.

## Step 4 · Compare

The 1994 Global Strategy is the comparison worth making, because it is a dated policy
intervention with a stated aim: a more representative and balanced List.

In [ ]:
STRATEGY_YEAR = 1994          # YOUR DECISION — and justify it

pre = df[df["year_inscribed"] < STRATEGY_YEAR]
post = df[df["year_inscribed"] >= STRATEGY_YEAR]
yrs_pre = STRATEGY_YEAR - df["year_inscribed"].min()
yrs_post = df["year_inscribed"].max() - STRATEGY_YEAR + 1

print(f"{'':<26}{'before 1994':>14}{'1994 onward':>14}")
print("-" * 56)
print(f"{'sites':<26}{len(pre):>14}{len(post):>14}")
print(f"{'years covered':<26}{yrs_pre:>14}{yrs_post:>14}")
print(f"{'sites per year':<26}{len(pre)/yrs_pre:>14.2f}{len(post)/yrs_post:>14.2f}")
print(f"{'countries represented':<26}{pre['country'].nunique():>14}{post['country'].nunique():>14}")
print()
print("category mix")
for cat in ["Cultural", "Natural", "Mixed"]:
    a = int((pre["category"] == cat).sum()); b = int((post["category"] == cat).sum())
    print(f"   {cat:<12}{a:>6} ({a/max(len(pre),1):>5.1%}) {'':<3}{b:>6} ({b/max(len(post),1):>5.1%})")
print()
print("Read those two blocks separately. The RATE barely moved. The MIX inverted.")
print("A chapter that reports only one of them has missed the finding.")

## Step 5 · Test

Two tests, and a lesson about which one your sample size permits.

In [ ]:
from scipy.stats import chi2_contingency, fisher_exact

df["era"] = np.where(df["year_inscribed"] < STRATEGY_YEAR, "pre-1994", "1994 on")
full = pd.crosstab(df["era"], df["category"])
print("observed counts")
print(full.to_string())
print()

chi2, p, dof, expected = chi2_contingency(full)
print("expected counts under independence")
print(pd.DataFrame(expected, index=full.index, columns=full.columns)
      .round(1).to_string())
print()
small = (expected < 5).sum()
print(f"cells with an expected count below 5: {small} of {expected.size}")
print()
if small:
    print("Chi-square is unreliable when expected counts fall below 5, and 'Mixed'")
    print("has only five sites in total. Reporting a chi-square here would be a")
    print("textbook error that a reviewer will catch. Collapse or use an exact test.")

In [ ]:
# Collapse to a 2x2 the data can actually support: Cultural vs everything else.
df["is_cultural"] = (df["category"] == "Cultural")
tab = pd.crosstab(df["era"], df["is_cultural"])
tab.columns = ["Natural or Mixed", "Cultural"]
print(tab.to_string())
print()

odds, p_fisher = fisher_exact(tab.values)
chi2b, p_chi, dofb, expb = chi2_contingency(tab)
phi = np.sqrt(chi2b / tab.values.sum())

print(f"Fisher's exact test   odds ratio {odds:.3f}   p = {p_fisher:.4f}")
print(f"chi-square {chi2b:.2f}   p = {p_chi:.4f}   phi = {phi:.3f}")
print(f"minimum expected count now: {expb.min():.1f}")
print()
print("Fisher's exact is the honest choice at this sample size; the chi-square is")
print("shown only so you can see they agree. Report Fisher, report the odds ratio,")
print("and report that you chose it because of the expected counts.")
print()
pre_share = pre["category"].eq("Cultural").mean()
post_share = post["category"].eq("Cultural").mean()
print(f"Cultural share before 1994: {pre_share:.1%}")
print(f"Cultural share 1994 onward: {post_share:.1%}")
print(f"change: {(post_share-pre_share)*100:+.1f} percentage points")

**YOUR DECISION.** What does this show about the Global Strategy?

The rate of African inscription did not rise. The composition shifted markedly toward
Cultural sites. Those are two different facts and they support different arguments:

- The Strategy **worked**, in that Africa's cultural heritage is now being inscribed
  where before the List treated the continent largely as landscape and wildlife.
- The Strategy **did not work**, in that the continent's total representation did not
  improve.

Both readings are defensible from your numbers. Choose one, argue it, and acknowledge
the other. That is what §6 of the chapter is for.

## Step 6 · Show

One chart. The criteria profile is the descriptive backbone, and the composition shift
is the analytical finding — so show both, and label the criteria with what they
actually claim rather than with roman numerals a reader has to look up.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

vals = [counts.get(c, 0) for c in ORDER]
colours = ["#9c2c1f"] * 6 + ["#2b5c50"] * 4      # cultural vs natural criteria
labels = [f"({c}) {MEANING[c][:26]}" for c in ORDER]
ax1.barh(labels[::-1], vals[::-1], color=colours[::-1])
ax1.set_xlabel("sites inscribed under this criterion")
ax1.set_title("What the List says African heritage is")
ax1.tick_params(labelsize=8)

eras = ["pre-1994", "1994 on"]
cats = ["Cultural", "Natural", "Mixed"]
bottom = np.zeros(2)
palette = {"Cultural": "#9c2c1f", "Natural": "#2b5c50", "Mixed": "#855f16"}
for cat in cats:
    v = np.array([int(((df["era"] == e) & (df["category"] == cat)).sum()) for e in eras], float)
    share_v = v / np.array([len(pre), len(post)], float) * 100
    ax2.bar(eras, share_v, bottom=bottom, label=cat, color=palette[cat])
    for i, (s, b) in enumerate(zip(share_v, bottom)):
        if s > 4:
            ax2.text(i, b + s/2, f"{s:.0f}%", ha="center", va="center",
                     color="white", fontsize=9)
    bottom += share_v
ax2.set_ylabel("share of inscriptions in the period (%)")
ax2.set_title(f"Composition before and after the {STRATEGY_YEAR} Global Strategy")
ax2.legend(loc="lower right", fontsize=8)

plt.tight_layout()
print(f"CAPTION. All {len(df)} African World Heritage sites inscribed "
      f"{df['year_inscribed'].min()}-{df['year_inscribed'].max()}; no filtering applied.")
print("Left: how often each inscription criterion is invoked; a site may be inscribed")
print("under several, so counts exceed the number of sites. Criteria (i)-(vi) are the")
print("cultural criteria (dark red) and (vii)-(x) the natural ones (green).")
print(f"Right: category composition before and from {STRATEGY_YEAR}, as a share of")
print(f"inscriptions in each period (n = {len(pre)} and {len(post)}).")

## Step 7 · The qualitative half

Your descriptions are **38 to 73 words** long, median 60. That is very short and very
uniform, which tells you they are standardised summaries rather than free prose.

**Do not topic-model them.** 115 documents of 60 words is far below what LDA needs; it
will return topics that look meaningful and are noise. Keyness plus close reading is
the honest instrument at this size.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def keyness(target_docs, ref_docs, min_count=3, top=12):
    """Log-likelihood keyness. Small corpus, so keep the frequency floor low."""
    vec = CountVectorizer(token_pattern=r"[a-z]{3,}", lowercase=True,
                          stop_words="english")
    X = vec.fit_transform(list(target_docs) + list(ref_docs))
    vocab = np.array(vec.get_feature_names_out())
    a = np.asarray(X[:len(target_docs)].sum(axis=0)).ravel()
    b = np.asarray(X[len(target_docs):].sum(axis=0)).ravel()
    na, nb = a.sum(), b.sum()
    out = []
    for i, w in enumerate(vocab):
        if a[i] + b[i] < min_count:
            continue
        ea = na * (a[i] + b[i]) / (na + nb); eb = nb * (a[i] + b[i]) / (na + nb)
        ll = 0.0
        if a[i]: ll += a[i] * np.log(a[i] / ea)
        if b[i]: ll += b[i] * np.log(b[i] / eb)
        ll *= 2
        if a[i] / na > b[i] / nb:
            out.append((w, int(a[i]), int(b[i]), ll))
    out.sort(key=lambda t: -t[3])
    return out[:top]

cultural = df[df["category"] == "Cultural"]["description"]
natural = df[df["category"] == "Natural"]["description"]

print("Words distinctive of CULTURAL site descriptions:")
for w, a_, b_, ll in keyness(cultural, natural):
    print(f"   {w:<18}{a_:>4} vs {b_:>4}   LL {ll:>6.1f}")
print()
print("Words distinctive of NATURAL site descriptions:")
for w, a_, b_, ll in keyness(natural, cultural):
    print(f"   {w:<18}{a_:>4} vs {b_:>4}   LL {ll:>6.1f}")
print()
print("LL >= 6.63 is p<.01 and >= 10.83 is p<.001 — but with a corpus this small,")
print("treat these as a reading list rather than as significance. The point is to")
print("find which words to look at, not to test a hypothesis.")

In [ ]:
# The five Mixed sites. Read every one — this is the heart of the chapter.
mixed = df[df["category"] == "Mixed"]
print(f"ALL {len(mixed)} MIXED SITES\n")
for _, r in mixed.iterrows():
    print(f"{r['site_name']}  ({r['country']}, {r['year_inscribed']})")
    print(f"   criteria: {r['criteria']}")
    print(f"   {' '.join(str(r['description']).split())}")
    print(f"   {r['url']}")
    print()

In [ ]:
# Candidates for misclassification: Cultural sites whose descriptions read like
# the Mixed ones. Cosine similarity on TF-IDF, used to SELECT what to read.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(token_pattern=r"[a-z]{3,}", stop_words="english")
X = vec.fit_transform(df["description"])
mixed_idx = df.index[df["category"] == "Mixed"]
mixed_centroid = np.asarray(X[df.index.get_indexer(mixed_idx)].mean(axis=0))

sims = cosine_similarity(X, mixed_centroid).ravel()
df["mixed_similarity"] = sims

cand = (df[df["category"] == "Cultural"]
        .sort_values("mixed_similarity", ascending=False)
        .head(8)[["site_name", "country", "criteria", "mixed_similarity"]])
print("CULTURAL sites whose descriptions most resemble the Mixed ones:\n")
print(cand.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print("These are your close-reading sample. The similarity score does not show they")
print("are misclassified — it shows which ones are worth arguing about. Read the")
print("descriptions and the criteria, then make the argument in prose.")

### The coding scheme

Read the eight candidates alongside the five Mixed sites. For each candidate, decide:

| Label | Definition |
|---|---|
| `CLEARLY_CULTURAL` | The natural setting is context only; the claim is about built or intangible heritage |
| `ARGUABLY_MIXED` | Natural and cultural values are described as interdependent |
| `NATURE_AS_BACKDROP` | Landscape appears but only as scenery for a cultural claim |
| `UNCLEAR` | The description is too short to judge |

**Two coders, independently.** Then report κ. If `ARGUABLY_MIXED` is common, you have
a quantified basis for the chapter's central claim — that the binary is doing work the
evidence does not support.

In [ ]:
from sklearn.metrics import cohen_kappa_score
from collections import Counter as C2

# Replace with your real codes after both coders have read the eight.
coder_a = ["ARGUABLY_MIXED","CLEARLY_CULTURAL","ARGUABLY_MIXED","NATURE_AS_BACKDROP",
           "ARGUABLY_MIXED","CLEARLY_CULTURAL","UNCLEAR","ARGUABLY_MIXED"]
coder_b = ["ARGUABLY_MIXED","CLEARLY_CULTURAL","NATURE_AS_BACKDROP","NATURE_AS_BACKDROP",
           "ARGUABLY_MIXED","CLEARLY_CULTURAL","UNCLEAR","CLEARLY_CULTURAL"]

kappa = cohen_kappa_score(coder_a, coder_b)
raw = float(np.mean([x == y for x, y in zip(coder_a, coder_b)]))
print(f"raw agreement {raw:.2f}   Cohen's kappa {kappa:.3f}")
print(f"distribution (coder A): {dict(C2(coder_a))}\n")
print("Disagreements:")
for i, (x, y) in enumerate(zip(coder_a, coder_b), 1):
    if x != y:
        print(f"   candidate {i}: {x} vs {y}")
print()
print("With only eight items, kappa is unstable — a single disagreement moves it a")
print("long way. Report the number AND the sample size, and do not lean on it as")
print("hard as a study with two hundred items could.")

## Step 8 · Limits

**What this analysis supports**

- Statements about the 115 African sites on the List as at the export date.
- The criteria profile, and the concentration of sites across states.
- A change in category composition before and after 1994, at the odds ratio reported
  by Fisher's exact test.

**What it does not support**

- Any claim about sites **not** on the List. Nominations that failed, were withdrawn,
  or were never made leave no trace here — and they are where the argument about
  representation actually lives.
- Any claim that the Global Strategy **caused** the compositional shift. You have a
  before and an after, not a counterfactual. Other things changed in 1994 too.
- Any comparison with the global List. That needs the full dataset, which this is not.
- Any claim that a specific site is misclassified. The similarity scores select
  reading; the argument has to be made in prose and defended.
- Statistical claims about `Mixed` as a category. Five sites cannot support a test.

**YOUR DECISION.** Add two more sentences this data does not support.

---

# Step 9 · The chapter template

Fill the gaps, then rewrite in your own voice. **Length: 6,000–8,000 words.**

---

## §1 Introduction — *about 800 words*

> The World Heritage List is the most visible instrument of international heritage
> recognition, and its categories ______________ . This chapter examines
> **_____ African sites inscribed between _____ and _____**, and argues that
> ______________ .

*Write last.*

---

## §2 The standard and its critics — *about 1,200 words*

The 1972 Convention, the ten criteria, the Cultural/Natural/Mixed categories, the
Operational Guidelines, and the 1994 Global Strategy. Then the critical literature on
Eurocentrism in heritage designation and on the cultural landscapes category
introduced in 1992.

> Sites are inscribed under criteria (i)-(x), of which ______________ are cultural
> and ______________ natural. Category follows from ______________ .
> The Global Strategy, adopted in ______________ , aimed to ______________ .

---

## §3 Data — *about 900 words*

> The dataset comprises **_____ sites** in **_____ states**, inscribed
> **_____–_____**: **_____ Cultural, _____ Natural, _____ Mixed**.
> It contains **no missing values**, which is a property of the nomination process
> rather than of the data: ______________ .

Then the absence audit, which for this chapter is about expressibility:

> The schema cannot record ______________ , ______________ or ______________ .
> The most consequential is ______________ , because ______________ .

---

## §4 Method — *about 700 words*

> Criteria were parsed from ______________ . Inscriptions were divided at
> ______________ , justified by ______________ .
> Association was tested with **Fisher's exact test** rather than chi-square,
> because ______________ .
> Descriptions were compared by log-likelihood keyness; topic modelling was **not**
> used, because ______________ .
> Eight candidate sites were selected by ______________ and coded independently by
> two readers.

**Both negative decisions belong here.** Choosing Fisher over chi-square, and refusing
to topic-model, are the two sentences that will tell a reviewer you understood your
sample size.

---

## §5 Findings — *about 1,800 words*

**§5.1 What the List says African heritage is**

> Criterion (iii), testimony to a cultural tradition, carries **_____** sites, and
> criterion (x), biodiversity, **_____**. Criterion (i), human creative genius,
> carries **_____**.

**§5.2 The Global Strategy shifted composition, not volume**

> Inscription ran at **_____ sites per year** before 1994 and **_____** after.
> The Cultural share moved from **_____%** to **_____%**
> (Fisher's exact, odds ratio _____, p = _____).

**§5.3 Concentration**

> **_____ of _____ states** hold a single site; **_____** holds twelve.

**§5.4 The binary under pressure** *(qualitative)*

> Only **_____ of _____** sites are inscribed as Mixed. Eight Cultural sites whose
> descriptions most resemble them were read closely (κ = _____, n = 8);
> **_____** were judged ______________ .

**Figure 1** after §5.1 or §5.2, captioned from Step 6.

---

## §6 Discussion — *about 1,400 words*

> The compositional shift is consistent with ______________ , and inconsistent with
> ______________ . Whether the Global Strategy succeeded depends on
> ______________ .
> The scarcity of Mixed inscriptions suggests ______________ .

The counter-practice:

> The cultural landscapes category, introduced in ______________ , responds by
> ______________ . Its limits are ______________ .

Also worth naming: the Africa 2009 programme, the African World Heritage Fund, and
national tentative-list practice.

---

## §7 Limitations · §8 Conclusion · §9 References · §10 Data and code

In [ ]:
print("=" * 74)
print("DRAFT SENTENCES — your numbers already placed")
print("=" * 74)

cat = df["category"].value_counts()
per_c = df["country"].value_counts()

print("""
§3 DATA
  The dataset comprises {n} African World Heritage sites in {c} states, inscribed
  between {y0} and {y1}: {cu} Cultural, {na} Natural and {mi} Mixed. It contains no
  missing values in any field. That completeness is a property of the nomination
  process rather than of the data - a dossier is refused until every required field
  is supplied - and so the absence audit for this chapter concerns what the schema
  cannot express rather than what it failed to record.
""".format(n=len(df), c=df["country"].nunique(), y0=int(df["year_inscribed"].min()),
           y1=int(df["year_inscribed"].max()), cu=int(cat.get("Cultural",0)),
           na=int(cat.get("Natural",0)), mi=int(cat.get("Mixed",0))))

print("""§5.1 WHAT THE LIST SAYS AFRICAN HERITAGE IS
  Criterion (iii), testimony to a cultural tradition, carries {c3} sites and
  criterion (x), biodiversity and threatened species, {c10}. Criterion (i), human
  creative genius, carries {c1}. Sites are inscribed under a mean of {m:.2f}
  criteria, in {k} distinct combinations.
""".format(c3=counts.get("iii",0), c10=counts.get("x",0), c1=counts.get("i",0),
           m=df["n_criteria"].mean(), k=df["criteria"].nunique()))

print("""§5.2 THE GLOBAL STRATEGY SHIFTED COMPOSITION, NOT VOLUME
  African inscription ran at {rp:.2f} sites per year before 1994 and {ro:.2f} from
  1994 onward. Over the same division the Cultural share of inscriptions moved from
  {sp:.1%} to {so:.1%} (Fisher's exact test, odds ratio {odds:.2f}, p = {pf:.4f}).
  The Strategy is therefore associated with a change in what is inscribed from
  Africa, and not with a change in how much.
""".format(rp=len(pre)/yrs_pre, ro=len(post)/yrs_post,
           sp=pre["category"].eq("Cultural").mean(),
           so=post["category"].eq("Cultural").mean(), odds=odds, pf=p_fisher))

print("""§5.3 CONCENTRATION
  {one} of {tot} states hold a single site each, while {top} holds {topn}. The mean
  is {mean:.2f} sites per state, a figure no state actually has.
""".format(one=int((per_c==1).sum()), tot=len(per_c), top=per_c.index[0],
           topn=int(per_c.iloc[0]), mean=len(df)/df["country"].nunique()))

print("""§5.4 THE BINARY UNDER PRESSURE
  Only {mi} of {n} sites are inscribed as Mixed. Eight Cultural sites whose
  descriptions most closely resemble the Mixed group were read independently by two
  coders (kappa = {k:.3f}, n = 8).
""".format(mi=int(cat.get("Mixed",0)), n=len(df), k=kappa))
print("=" * 74)
print("Left for you: what it MEANS, what it cannot support, the counter-practice.")

### Three mistakes that sink first chapters

**Reporting completeness as data quality.** "The dataset is complete and well
maintained" throws away your best opening. Completeness is manufactured by the
nomination process, and saying so is the chapter's first move.

**Using chi-square with five Mixed sites.** The expected counts are below five and a
reviewer will check. You ran Fisher's exact instead — say why in §4.

**Claiming the Global Strategy caused the shift.** You have a before and an after.
Causal language here is the easiest thing in the chapter to attack, and the honest
version — "associated with", plus a note on what a counterfactual would require — is
stronger.

---

# Step 10 · Dividing the work

More than five people, one chapter. Divide by **expertise**, not by paragraph count.

## The roles

| # | Role | Owns | Expertise it draws on | Hands over |
|---|---|---|---|---|
| 1 | **Corpus &amp; standard** | §3, the expressibility audit | Heritage policy, the Operational Guidelines | What the schema can and cannot say |
| 2 | **Analysis** | §4, §5.1–5.3 | Statistics, computation | The criteria profile, the Fisher test, concentration |
| 3 | **Coder A** | §5.4, jointly | Heritage studies, landscape geography | 8 independently assigned codes |
| 4 | **Coder B** | §5.4, jointly | Heritage studies, landscape geography | 8 independently assigned codes |
| 5 | **Policy &amp; history** | §2, §6 | The 1972 Convention, the Global Strategy, the critics | The argument the chapter joins |
| 6 | **Visualisation &amp; communication** | Figure 1, §8 | Design, editing | A criteria chart legible without the text |
| 7 | **Integration editor** | §1, §7, references | Editorial judgement | One voice, and a chapter that ends |

Role 5 matters unusually here: the Global Strategy, the 1992 cultural-landscapes
category and the Africa 2009 programme are specific instruments with documents behind
them. A chapter that treats them vaguely will not survive review.

## Why coders 3 and 4 are two people

A **method requirement.** Judging whether a site's description makes natural and
cultural values interdependent is exactly the kind of reading two careful people do
differently. Kappa converts that into a reportable figure.

**With only eight items, be honest about instability.** One disagreement moves kappa
substantially. Report the sample size next to the coefficient every time.

## The order things happen in

```
   Corpus & standard ──┐
                       ├──► Analysis ──► Visualisation ──┐
   Coders A + B ───────┘                                 ├──► Integration
                                                         │
   Policy & history ─────────────────────────────────────┘
```

## Combining the drafts

**One person edits for voice, and everyone accepts the edit.**

**Agree the terms in writing first.** For this chapter: *inscription* or *listing*?
*Site* or *property*? UNESCO's own term is **property**, and using it signals you have
read the Operational Guidelines. Decide, then be consistent.

## Declaring who did what

Use **CRediT**.

In [ ]:
TEAM = {
    "Corpus & standard":       ("________________", "data curation, investigation"),
    "Analysis":                ("________________", "formal analysis, software, methodology"),
    "Coder A":                 ("________________", "investigation, validation"),
    "Coder B":                 ("________________", "investigation, validation"),
    "Policy & history":        ("________________", "conceptualisation, writing - original draft"),
    "Visualisation & comms":   ("________________", "visualisation, writing - review and editing"),
    "Integration editor":      ("________________", "writing - review and editing, supervision"),
}
print("AUTHOR CONTRIBUTIONS (CRediT)")
print()
for role, (name, credit) in TEAM.items():
    print(f"  {name}: {credit}.")
    print(f"      [{role}]")
print()
print("Fill in names, delete the bracketed labels, place after the conclusion.")
print("Agree authorship order EARLY.")

### What goes wrong, and how to see it coming

**The sample size gets forgotten.** 115 sites, 5 Mixed, 8 coded candidates. Every
claim has to be sized to that, and the notebook stops you three times for a reason.

**Policy swallows the chapter.** The Convention's history is long and it is not your
finding. Role 5 supplies §2 and §6; the evidence stays yours.

**The coders talk.** Fatal to the kappa, and with eight items there is nowhere to hide.

**Nobody owns the ending.** Role 7 owns §8 and the decision that it is finished.

## What to hand in

1. **One page**: the finding, the method, the uncertainty, and the limits list.
2. **One chart**, captioned, saying what you filtered.
3. **The coding sheet** with both coders' labels, the kappa, and the sample size.

### Turning this into the chapter

Your spine: a standard that manufactures completeness as a condition of entry, a
criteria profile that shows what the List is willing to see as African heritage, a
dated policy intervention that changed composition but not volume, and five sites out
of 115 permitted to be both cultural and natural.

What it needs from you is **§6** — what a binary costs when applied to landscapes it
was not built for — and the counter-practice. The cultural landscapes category is the
obvious one, and asking why so few African sites use it is a chapter in itself.